In [1]:
import re
import unicodedata
import pandas as pd
from pathlib import Path

In [2]:
mis_form_path = "../../backend/source/forms"
flow_form_path = "./output/flow_forms"
flow_ids = {
    "8520967": 1749634736797, # WTP
    "17260923": 1748903240763, # WWTP
    "27040920": 1749611049520, # SPS (Pump Station)
    "1520924": 1749623934933, # EPS Water quality
    "5530933": 1749623934933, # EPS Project Construction
    "2490944": 1749621221728, # RWS
}
meta_config = {
    "flow": [
        "name",
        "surveyId",
        "surveyGroupId",
        "version",
        "app",
        ["questionGroup", "heading"],
        ["questionGroup", "repeatable"],
    ],
    "mis": [
        "id",
        "form",
        "type",
        "version",
        ["question_groups", "label"],
        ["question_groups", "repeatable"],
    ],
}

In [3]:
def flatten_flow_form_pd(
    json_path: Path,
    record_path: str = "questionGroup.question",
    meta_type: str = "flow",
) -> pd.DataFrame:
    if meta_type not in meta_config:
        raise ValueError(f"meta_type must be one of {list(meta_config.keys())}")
    # read top-level JSON as a pandas Series (one object)
    j = pd.read_json(json_path, typ="series").to_dict()

    # normalize nested questions: record_path points to question lists inside questionGroup
    df = pd.json_normalize(
        j,
        record_path=record_path.split("."),
        meta=meta_config[meta_type],
        meta_prefix=f"{meta_type}_",
        errors="ignore"
    )
    # rename question id column for clarity
    if "id" in df.columns:
        df = df.rename(
            columns={
                "id": f"{meta_type}_question_id"
            }
        )

    # rename group meta columns for clarity
    if "flow_questionGroup.heading" in df.columns:
        df = df.rename(
            columns={
                "flow_questionGroup.heading": "flow_question_group",
                "flow_questionGroup.repeatable": "flow_question_group_repeatable"
            }
        )
    if "mis_question_groups.label" in df.columns:
        df = df.rename(
            columns={
                "mis_question_groups.label": "mis_question_group",
                "mis_question_groups.repeatable": "mis_question_group_repeatable"
            }
        )
    return df

In [4]:
def _normalize_alpha_only(val):
    if pd.isna(val):
        return ""
    s = str(val)
    s = unicodedata.normalize("NFKD", s)
    s = s.encode("ascii", "ignore").decode("ascii")  # remove accents
    s = s.lower()
    # remove anything that's not a-z (keeps only letters)
    s = re.sub(r"[^a-z]+", "", s)
    return s

In [5]:
# load flow forms mapping
for json_form in filter(
    lambda f: f.suffix == ".json",
    Path(flow_form_path).iterdir(),
):
    print(f"Loading form from {json_form}")
    # get filename and extract form id in prefix
    # remove path and suffix
    flow_form_id = json_form.name.split("_")[0]
    print(f"Form ID: {flow_form_id}")
    
    flow_questions_df = flatten_flow_form_pd(json_form)
    # Get MIS form id from mapping
    mis_form_id = flow_ids.get(flow_form_id)
    # find JSON form in MIS forms folder that includes the MIS form id in the filename
    mis_form_json = next(filter(
        lambda f: f.suffix == ".json" and str(mis_form_id) in f.name,
        Path(mis_form_path).iterdir(),
    ), None)
    if mis_form_json is None:
        print(f"No matching MIS form found for Flow form ID {flow_form_id} and MIS form ID {mis_form_id}")
        continue
    print(f"Matching MIS form found: {mis_form_json.name}")
    mis_questions_df = flatten_flow_form_pd(
        json_path=mis_form_json,
        record_path="question_groups.questions",
        meta_type="mis",
    )

    # find JSON form in MIS forms folder that have parent_id equal to the MIS form id
    # load and flatten these as well, then append to mis_questions_df
    for child_form_json in filter(
        lambda f: f.suffix == ".json",
        Path(mis_form_path).iterdir(),
    ):
        # read top-level JSON as a pandas Series (one object)
        j = pd.read_json(child_form_json, typ="series").to_dict()
        if j.get("parent_id") == mis_form_id:
            print(f"Loading child MIS form: {child_form_json.name}")
            child_mis_questions_df = flatten_flow_form_pd(
                json_path=child_form_json,
                record_path="question_groups.questions",
                meta_type="mis",
            )
            mis_questions_df = pd.concat([mis_questions_df, child_mis_questions_df], ignore_index=True)

    # Normalize question text and group headings for robust matching
    flow_questions_df["_norm"] = flow_questions_df.get("text", "").apply(_normalize_alpha_only)
    mis_questions_df["_norm"] = mis_questions_df.get("label", "").apply(_normalize_alpha_only)

    flow_questions_df["_group_norm"] = flow_questions_df.get("flow_question_group", "").apply(_normalize_alpha_only)
    mis_questions_df["_group_norm"] = mis_questions_df.get("mis_question_group", "").apply(_normalize_alpha_only)

    rows = []
    # iterate every unique flow question row grouped by surveyId, group, question id
    for (survey_id, group, qid), grp in flow_questions_df.groupby(
        ["flow_surveyId", "flow_question_group", "flow_question_id"], dropna=False
    ):
        # there should be exactly one row per group (but handle multiple if present)
        # take the first representative row
        rep = grp.iloc[0]
        f_text = rep.get("text", "")
        f_norm = rep.get("_norm", "")
        f_group_norm = rep.get("_group_norm", "")

        # primary candidate: both normalized text and normalized group match
        candidates = mis_questions_df[
            (mis_questions_df["_norm"] == f_norm) &
            (mis_questions_df["_group_norm"] == f_group_norm)
        ]

        # fallback 1: match by normalized text only (ignore group)
        if candidates.empty:
            candidates = mis_questions_df[mis_questions_df["_norm"] == f_norm]

        # fallback 2: match by normalized group only (ignore text)
        if candidates.empty and f_group_norm:
            candidates = mis_questions_df[mis_questions_df["_group_norm"] == f_group_norm]

        # collect aggregated values (semicolon-separated unique strings)
        def agg_col(df, col):
            if col not in df.columns or df.empty:
                return ""
            vals = df[col].dropna().astype(str).unique()
            return ";".join(vals)

        mis_id_val = agg_col(candidates, "mis_id")
        mis_qgroup_val = agg_col(candidates, "mis_question_group")
        mis_qid_val = agg_col(candidates, "mis_question_id")
        label_val = agg_col(candidates, "label")

        rows.append({
            "flow_surveyId": survey_id,
            "flow_question_group": group,
            "flow_question_label": f_text,
            "flow_question_id": qid,
            "mis_id": mis_id_val,
            "mis_question_group": mis_qgroup_val,
            "mis_question_label": label_val,
            "mis_question_id": mis_qid_val,
        })

    mapping_rf = pd.DataFrame(rows)

    # Optional: if you want to keep only rows with at least one MIS match, uncomment:
    # mapping_rf = mapping_rf[mapping_rf["mis_id"] != ""]

    # Guarantee uniqueness: drop duplicates if any (shouldn't be needed)
    mapping_rf = mapping_rf.drop_duplicates(
        subset=["flow_surveyId", "flow_question_group", "flow_question_id"],
        keep="first"
    ).reset_index(drop=True)

    # Save per-flow mapping CSV (same pattern used previously)
    output_csv = f"../../backend/source/akvo-flow/{flow_form_id}_forms_mapping.csv"
    mapping_rf.to_csv(output_csv, index=False)
    print(f"Saved {len(mapping_rf)} mapping rows to {output_csv}")
        

Loading form from output/flow_forms/17260923_WAF_Wastewater_Treatment_Plant_Inspection.json
Form ID: 17260923
Matching MIS form found: 5_1748903240763.prod.json
Loading child MIS form: 5_1748905550055.monitoring.prod.json
Loading child MIS form: 5_1748918946591.monitoring.prod.json
Saved 103 mapping rows to ../../backend/source/akvo-flow/17260923_forms_mapping.csv
Loading form from output/flow_forms/2490944_Rural_Water_Project_Inspection.json
Form ID: 2490944
Matching MIS form found: 3_1749621221728.prod.json
Loading child MIS form: 3_1749621962296.monitoring.prod.json
Loading child MIS form: 3_1749631041125.monitoring.prod.json
Saved 39 mapping rows to ../../backend/source/akvo-flow/2490944_forms_mapping.csv
Loading form from output/flow_forms/5530933_EPS_Projects_Construction_Monitoring.json
Form ID: 5530933
Matching MIS form found: 2_1749623934933.prod.json
Loading child MIS form: 2_1749624452908.monitoring.prod.json
Loading child MIS form: 2_1749632545233.monitoring.prod.json
Saved